# Generative LLM — `career_position` Annotation

**Model:** Qwen3 80B (LM Studio)  
**Granularity:** Fine-grained codes  
**Prompting:** zero-shot (set `N_SHOTS > 0` for few-shot)  

> LM Studio must be running at `LMSTUDIO_URL` before executing inference.

## 1. Imports & config

In [3]:
from dotenv import load_dotenv
load_dotenv()

import os
import re
import time
from tqdm.auto import tqdm
from openai import OpenAI

from corex_eval import evaluate, load_inputs, load_training_data, career_position_to_sector

# --- Config ---
LMSTUDIO_URL = "http://localhost:1234/v1"
MODEL_ID     = "qwen/qwen3-next-80b"   # ← verify with the "Discover models" cell below
N_SHOTS      = 0    # 0 = zero-shot; set to 3 or 5 for few-shot
SEED         = 42

## 2. Set up client & discover model IDs

In [4]:
client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")

# Verify MODEL_ID against what is actually loaded in LM Studio
print("Models available in LM Studio:")
try:
    for m in client.models.list():
        print(f"  {m.id}")
except Exception as e:
    print(f"  Could not connect to LM Studio at {LMSTUDIO_URL}: {e}")

Models available in LM Studio:
  qwen/qwen3-next-80b
  meta/llama-3.3-70b
  text-embedding-nomic-embed-text-v1.5
  deepseek-r1-distill-qwen-7b
  devstral-small-2505
  gemma-3-27b-it-qat
  openai/gpt-oss-20b


## 3. Load valid codes

In [5]:
train_df = load_training_data(
    task="annotation",
    variable="career_position",
    features=["job_description_label", "workplace_label", "career_position"],
)
train_df = train_df.dropna(subset=["job_description_label", "career_position"]).reset_index(drop=True)
valid_codes = sorted(train_df["career_position"].unique())
print(f"{len(valid_codes)} valid career position codes")

/tmp/pycharm_project_967/corex_eval/silver.py:146: UserWarning: [silver] Dropped 11 row(s) for variable 'career_position' whose case_ids are not present in the gold standard.
  warnings.warn(


## 4. Build prompt & prediction parser

In [6]:
def build_system_prompt(valid_codes):
    codes_str = "\n".join(f"  {c}" for c in valid_codes)
    return (
        "You are an expert political scientist coding political career histories "
        "using the CoREx codebook.\n\n"
        "Task: given a job description, assign the single most appropriate career position code.\n"
        "Respond with ONLY the exact code label as listed below — "
        'for example: "105 = Minister with portfolio (including European Commissioner)"\n'
        "Do not add any explanation, commentary, or extra formatting.\n\n"
        f"Valid career position codes:\n{codes_str}"
    )

def build_messages(job_description, workplace, system_prompt, shot_examples):
    user_content = f"Job title: {job_description}\nWorkplace: {workplace or 'unknown'}"
    messages = [{"role": "system", "content": system_prompt}]
    for desc, wp, code in shot_examples:
        messages.append({"role": "user", "content": f"Job title: {desc}\nWorkplace: {wp or 'unknown'}"})
        messages.append({"role": "assistant", "content": code})
    messages.append({"role": "user", "content": user_content})
    return messages

def parse_prediction(response_text, valid_codes):
    """Robustly map model response to the nearest valid code."""
    text = response_text.strip().strip("\"'")
    if text in valid_codes:
        return text
    code_map = {c.split(" = ", 1)[0].strip(): c for c in valid_codes if " = " in c}
    for code_num, full_code in code_map.items():
        if re.search(r"\b" + re.escape(code_num) + r"\b", text):
            return full_code
    text_lower = text.lower()
    for code in valid_codes:
        desc = code.split(" = ", 1)[1].lower() if " = " in code else code.lower()
        if desc in text_lower or text_lower in desc:
            return code
    return text

SYSTEM_PROMPT = build_system_prompt(valid_codes)
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

System prompt: 11866 chars


## 5. Few-shot examples (skipped when `N_SHOTS=0`)

In [7]:
if N_SHOTS > 0:
    shot_pool = (
        train_df.assign(sector=train_df["career_position"].map(career_position_to_sector))
        .groupby("sector", group_keys=False)
        .apply(lambda g: g.sample(1, random_state=SEED))
        .reset_index(drop=True)
    )
    shot_examples = (
        shot_pool.sample(min(N_SHOTS, len(shot_pool)), random_state=SEED)
        [["job_description_label", "workplace_label", "career_position"]]
        .values.tolist()
    )
    print(f"Using {len(shot_examples)} few-shot examples")
else:
    shot_examples = []
    print("Zero-shot mode")

Zero-shot mode


## 6. Load test inputs

In [8]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")
test_df.head()

,case_id,spell_index,job_description_label,career_position,workplace_label
0,002002,1,Minister of Infrastructure and Energy,105 = Minister with portfolio (including Europ...,Government of Albania
1,002002,2,General Director,514 = Regulatory Agencies: Transport,"ALBCONTROL, JSC"
2,002002,3,Public Relations Advisor and Director of the M...,"444 = Political employment, local level, e.g. ...",Municipality of Tirana
3,002015,1,Minister of Interior,105 = Minister with portfolio (including Europ...,Government of Albania
4,002015,3,Secretary General,128 = Other political executive triangle position,Socialist Party


## 7. Run inference

In [10]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    messages = build_messages(
        row["job_description_label"],
        row.get("workplace_label", ""),
        SYSTEM_PROMPT,
        shot_examples,
    )
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=0,
            max_tokens=64,
            seed=SEED,
        )
        raw = response.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
        time.sleep(0.1)
    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

,case_id,spell_index,predicted_code
0,002002,1,105 = Minister with portfolio (including Europ...
1,002002,2,702 = Mid or low-level managing EU administrat...
2,002002,3,110 = Head of ministerial cabinet same ministr...
3,002015,1,105 = Minister with portfolio (including Europ...
4,002015,3,"432 = Party office / function, national level"


## 8. Evaluate

In [12]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
    granularity="fine",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/career/lmstudio_qwen3_80b_fine/config.yaml",
)

[corex_eval] Results submitted to GitHub register.
  Repo      : Tombellens/corex_eval
  File      : results/register.csv
  Task      : annotation / career_position
  Contributor: tom
  Timestamp : 2026-03-23T12:27:39.989627+00:00

──────────────────────────────────────────────────
  CoREx evaluation — annotation / career_position
──────────────────────────────────────────────────
  Cases evaluated : 3553
  Cases skipped   : 0
  Accuracy     : 0.176471
  Macro F1     : 0.104722
  Weighted F1  : 0.163911

  Per-country accuracy:
    ALB                            0.2075  (n=53)
    AUT                            0.1562  (n=96)
    BGR                            0.1020  (n=49)
    BIH                            0.2899  (n=69)
    CZE                            0.1971  (n=137)
    DEU                            0.1577  (n=298)
    DNK                            0.0950  (n=179)
    ESP                            0.1195  (n=251)
    EST                            0.1818  (n=77)
    FRA     